# Attention on a T4: naive vs PyTorch SDPA (math vs memory-efficient)

Companion to the card **flashattention-backward-results.html** (perf-4-attention-kv). The FlashAttention paper's
section 4.3 shows two things on an A100: tiled, recomputing attention is faster than standard attention, and its
memory grows **linearly** in the sequence length N instead of quadratically. This notebook checks the same shape on
the GPU you get on free Colab.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

What it runs (fp16, batch 2, 8 heads, head dim 64, N = 512 … 16,384, no mask, no dropout):

1. **naive**: `softmax(Q Kᵀ / √d) V` written out in PyTorch. It materialises the N×N scores and probabilities, and autograd keeps P for the backward pass.
2. **SDPA, MATH backend**: `torch.nn.functional.scaled_dot_product_attention` forced onto its reference C++ path. Also O(N²) memory.
3. **SDPA, EFFICIENT_ATTENTION backend**: the memory-efficient kernel (from xFormers). Tiled like FlashAttention, stores only per-row statistics, recomputes in the backward pass.
4. **SDPA, FLASH_ATTENTION backend**: needs compute capability 8.0 or newer (Ampere). A T4 is Turing, sm_75, so the notebook detects that and prints that it's unavailable.

For each one it measures **time** (CUDA events, after warm-up) and **extra peak memory** (`torch.cuda.max_memory_allocated` minus what the inputs already use),
for the **forward pass only** (what inference runs, under `torch.no_grad()`) and for **forward + backward** (what training runs). It checks correctness against an fp32 reference first,
then prints a table and a log-log plot.

Not yet run by the card's author. Colab T4 clocks vary, so treat the times as rough; the memory numbers should be steady.

In [ ]:
!nvidia-smi

In [ ]:
import math, torch
import torch.nn.functional as F
from torch.nn.attention import sdpa_kernel, SDPBackend

assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU"
dev = torch.device("cuda")
cap = torch.cuda.get_device_capability()
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(), f"| compute capability sm_{cap[0]}{cap[1]}")
print("total GPU memory: %.1f GiB" % (torch.cuda.get_device_properties(0).total_memory / 2**30))

B, H, D = 2, 8, 64          # batch, heads, head dim (fixed)
DTYPE = torch.float16       # T4 has no fast bf16
NS = [512, 1024, 2048, 4096, 8192, 16384]

def inputs(n, grad=False, dtype=DTYPE, seed=0):
    g = torch.Generator(device=dev).manual_seed(seed)
    q, k, v, do = (torch.randn(B, H, n, D, device=dev, dtype=dtype, generator=g) for _ in range(4))
    if grad:
        for t in (q, k, v):
            t.requires_grad_(True)
    return q, k, v, do

## The four implementations

`naive` is the paper's Algorithm 0 in PyTorch. The SDPA variants are the same function call, pinned to one backend with `sdpa_kernel`.
If a backend can't run these inputs on this GPU, PyTorch raises a `RuntimeError` ("No available kernel"); the notebook catches exactly that and records the backend as unavailable.

In [ ]:
def naive(q, k, v):
    s = (q * (1.0 / math.sqrt(q.shape[-1]))) @ k.transpose(-2, -1)   # N x N scores
    p = torch.softmax(s, dim=-1)                                    # N x N probabilities (kept for backward)
    return p @ v

def sdpa_with(backend):
    def f(q, k, v):
        with sdpa_kernel(backend):
            return F.scaled_dot_product_attention(q, k, v)
    return f

IMPLS = {
    "naive": naive,
    "sdpa MATH": sdpa_with(SDPBackend.MATH),
    "sdpa EFFICIENT": sdpa_with(SDPBackend.EFFICIENT_ATTENTION),
    "sdpa FLASH": sdpa_with(SDPBackend.FLASH_ATTENTION),
}

# Which backends can actually run here?
available = {}
for name, f in IMPLS.items():
    q, k, v, _ = inputs(256)
    try:
        f(q, k, v)
        torch.cuda.synchronize()
        available[name] = True
    except RuntimeError as e:
        available[name] = False
        print(f"{name}: unavailable on sm_{cap[0]}{cap[1]} -> {str(e).splitlines()[0][:120]}")
if not available["sdpa FLASH"] and cap < (8, 0):
    print("As expected: the FLASH_ATTENTION backend needs sm_80+ (Ampere or newer); this GPU is sm_%d%d." % cap)
print("available:", available)

## Correctness first

Reference: the naive formula in **fp32** at N = 1,024. Each fp16 implementation should match it to about 1e-3 on O and on the three gradients.

In [ ]:
n = 1024
q32, k32, v32, do32 = inputs(n, grad=True, dtype=torch.float32, seed=1)
o_ref = naive(q32, k32, v32)
o_ref.backward(do32)
ref = [o_ref.detach(), q32.grad, k32.grad, v32.grad]

print(f"{'impl':16s} {'max|dO|':>9s} {'max|dQ|':>9s} {'max|dK|':>9s} {'max|dV|':>9s}   (abs error vs fp32 naive)")
for name, f in IMPLS.items():
    if not available[name]:
        continue
    q, k, v, do = inputs(n, grad=True, dtype=DTYPE, seed=1)
    o = f(q, k, v)
    o.backward(do)
    errs = [(a.float() - b).abs().max().item() for a, b in zip([o.detach(), q.grad, k.grad, v.grad], ref)]
    print(f"{name:16s} " + " ".join(f"{e:9.2e}" for e in errs))

## Time and peak memory vs N

Predicted size of **one** N×N fp16 matrix for this shape = B·H·N²·2 bytes (our arithmetic, not a measurement):
8 MiB at 512, 32 MiB at 1k, 128 MiB at 2k, 512 MiB at 4k, 2 GiB at 8k, 8 GiB at 16k. The naive version holds a few of these at once,
so expect it (and MATH) to run out of the T4's ~15 GiB somewhere between 8k and 16k. Out-of-memory is recorded as `OOM`, not an error.

In [ ]:
def measure(f, n, backward, reps):
    q, k, v, do = inputs(n, grad=backward)
    def step():
        if backward:
            for t in (q, k, v):
                t.grad = None
            f(q, k, v).backward(do)
        else:
            with torch.no_grad():
                f(q, k, v)
    torch.cuda.synchronize()
    base = torch.cuda.memory_allocated()
    torch.cuda.reset_peak_memory_stats()
    step()                                    # warm-up 1 (also the memory measurement)
    torch.cuda.synchronize()
    peak_mib = (torch.cuda.max_memory_allocated() - base) / 2**20
    step()                                    # warm-up 2
    start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(reps):
        step()
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / reps, peak_mib

results = {}   # (impl, mode, n) -> (ms, MiB) or None for OOM
for mode, backward in [("fwd", False), ("fwd+bwd", True)]:
    for name, f in IMPLS.items():
        if not available[name]:
            continue
        for n in NS:
            reps = 20 if n <= 2048 else 5
            try:
                results[(name, mode, n)] = measure(f, n, backward, reps)
            except torch.cuda.OutOfMemoryError:
                results[(name, mode, n)] = None
            torch.cuda.empty_cache()
            r = results[(name, mode, n)]
            print(f"{mode:8s} {name:15s} N={n:6d}  " + ("OOM" if r is None else f"{r[0]:9.2f} ms  {r[1]:9.1f} MiB"))

In [ ]:
names = [nm for nm in IMPLS if available[nm]]
for mode in ["fwd", "fwd+bwd"]:
    print(f"\n{mode}: time (ms) / extra peak memory (MiB) on {torch.cuda.get_device_name()}, B={B}, H={H}, d={D}, fp16")
    print(f"{'N':>6s} " + " ".join(f"{nm:>24s}" for nm in names) + "   naive÷EFFICIENT (time, memory)")
    for n in NS:
        row = []
        for nm in names:
            r = results.get((nm, mode, n))
            row.append(f"{'OOM':>24s}" if r is None else f"{r[0]:10.2f} / {r[1]:10.1f}")
        a, b = results.get(("naive", mode, n)), results.get(("sdpa EFFICIENT", mode, n))
        ratio = f"{a[0]/b[0]:5.2f}x, {a[1]/max(b[1],1e-9):6.1f}x" if a and b else "-"
        print(f"{n:6d} " + " ".join(row) + "   " + ratio)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for col, mode in enumerate(["fwd", "fwd+bwd"]):
    for nm in names:
        pts = [(n, results[(nm, mode, n)]) for n in NS if results.get((nm, mode, n))]
        if not pts:
            continue
        xs = [p[0] for p in pts]
        axes[0][col].plot(xs, [p[1][0] for p in pts], marker="o", label=nm)
        axes[1][col].plot(xs, [p[1][1] for p in pts], marker="o", label=nm)
    axes[0][col].set_title(f"{mode}: time (ms)")
    axes[1][col].set_title(f"{mode}: extra peak memory (MiB)")
    for ax in (axes[0][col], axes[1][col]):
        ax.set_xscale("log", base=2); ax.set_yscale("log"); ax.set_xlabel("sequence length N")
        ax.grid(True, which="both", alpha=0.3); ax.legend()
fig.suptitle(f"{torch.cuda.get_device_name()}  B={B} H={H} d={D} fp16  (missing points = OOM)")
fig.tight_layout(); plt.show()

## What to look for

- **Memory slope.** On a log-log plot, a slope of 2 means quadratic (doubling N quadruples memory) and a slope of 1 means linear. The paper's A100 numbers (Table 21, batch 16, 8 heads, d 64): PyTorch 1,184 → 4,416 → 17,024 MB for N = 1k → 2k → 4k (×3.7, ×3.9), FlashAttention 209 → 418 → 836 MB (×2.0, ×2.0).
- **fwd+bwd vs fwd.** The naive version has to *keep* P for the backward pass, so its fwd+bwd memory is higher than its fwd memory under `no_grad`. The efficient kernel keeps only per-row statistics.
- **Speed.** The paper says FlashAttention's speedup is smaller on a T4 than on an A100, because the T4's smaller SRAM forces smaller blocks (App E.5). Whatever ratio you see here is a T4 result for this shape, not the paper's A100 number.

## Reference numbers from the source

Dao et al., *FlashAttention* (arXiv 2205.14135), Table 11, A100 40 GB, batch 16, 8 heads, d 64, with dropout and masking, fwd+bwd ms:

| N | 128 | 512 | 1024 | 2048 | 4096 | 8192 | 16384 |
|---|---|---|---|---|---|---|---|
| PyTorch | 0.84 | 2.35 | 8.29 | 31.75 | 124.19 | OOM | OOM |
| FlashAttention | 0.43 | 0.95 | 2.55 | 9.56 | 37.49 | 147.75 | 586.61 |

## Try this

- Set `B = 16` to match the paper's batch. Naive and MATH run out of memory much earlier (one N×N matrix is 8× bigger).
- Pass `is_causal=True` to `scaled_dot_product_attention` (and add a causal mask to `naive`). The efficient kernel skips masked blocks; how much of the time goes away?
- On a Colab A100 or L4 (sm_80+/sm_89), rerun: `sdpa FLASH` becomes available. Compare it with `sdpa EFFICIENT` on the same shape.